# Titanic Survival Predictor — Training (Google Colab version)

This notebook does the exact same thing as `training/train.py` in the repo, but broken into cells so you can run it step by step in Google Colab and see what happens at each stage.

**How to use this:**
1. Run each cell in order (click the cell, press Shift+Enter, or use "Run all" from the Runtime menu).
2. Read the printed output as you go — it explains what happened.
3. The last cell downloads the trained model file to your computer. You'll use that file to run the backend locally afterwards.

## 1. Install the libraries we need

Colab already has most of these, but this makes sure the versions match (especially `scikit-learn` — it must match the version used by the FastAPI backend in `app/backend/requirements.txt`, otherwise the backend won't be able to load the saved model file).

In [ ]:
!pip install -q pandas scikit-learn==1.6.1 seaborn joblib

## 2. Load the Titanic dataset

We use `seaborn`'s built-in copy of the classic Titanic dataset, so there's nothing to upload or download manually.

In [ ]:
import pandas as pd
import seaborn as sns

FEATURE_COLUMNS = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
TARGET_COLUMN = "survived"

df = sns.load_dataset("titanic")
df = df[FEATURE_COLUMNS + [TARGET_COLUMN]]

print(f"Loaded {len(df)} passengers.")
df.head()

## 3. Build the preprocessing + model pipeline

Real-world data is messy — some passengers are missing an age or a port of embarkation. This pipeline fills those gaps in automatically (with the median age / most common port), converts text columns like `sex` and `embarked` into numbers the model can use, and then trains a `RandomForestClassifier` on the result — all as a single reusable object.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked"]

numeric_transformer = SimpleImputer(strategy="median")
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)

pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", model)])
pipeline

## 4. Split the data and train

We hold back 20% of passengers as a "test set" the model never sees during training, so we can check afterwards how well it does on data it hasn't memorized.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline.fit(X_train, y_train)
print("Training complete.")

## 5. Evaluate the model

- **Accuracy**: the percentage of test passengers the model classified correctly.
- **Confusion matrix**: rows are what actually happened, columns are what the model predicted. The diagonal (top-left to bottom-right) is correct predictions.
- **Classification report**: precision/recall broken down by class.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = pipeline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}\n")
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Did not survive", "Survived"]))

## 6. Save and download the trained model

This saves the trained pipeline to a file called `titanic_model.joblib` and downloads it straight to your computer's Downloads folder.

Once it's downloaded, move it into your local copy of the repo at:

```
app/backend/model/titanic_model.joblib
```

That's the file the FastAPI backend loads when it starts up.

In [ ]:
import joblib

joblib.dump(pipeline, "titanic_model.joblib")
print("Saved titanic_model.joblib")

try:
    from google.colab import files
    files.download("titanic_model.joblib")
except ImportError:
    print("Not running in Colab — the file is saved in the current directory instead.")